# 02 Parse And Normalize Events

Цель этапа: разобрать `event_json`, сделать канонизацию и подготовить структурированное представление событий без coarse-схлопывания.

Вход:

- `data/interim/events_shard_*.parquet`
- `artifacts/manifests/ingest_manifest.json`

Выход:

- `data/interim/events_normalized_*.parquet`
- `artifacts/manifests/normalize_manifest.json`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/root/projects/bert4rec')

In [2]:
from src.normalization import load_ingest_manifest, load_normalize_config, normalize_parquet_shards

config = load_normalize_config(PROJECT_ROOT / "configs" / "normalization.yaml")
ingest_manifest = load_ingest_manifest(PROJECT_ROOT / config.input_manifest_path)

print("input shards:", ingest_manifest["num_shards"])
print("input rows:", ingest_manifest["rows_out"])
config

input shards: 74
input rows: 73063802


NormalizeConfig(input_manifest_path='artifacts/manifests/ingest_manifest.json', output_dir='data/interim', output_pattern='events_normalized_{shard_idx:05d}.parquet', manifest_path='artifacts/manifests/normalize_manifest.json', compression='zstd', row_group_size=100000, json=JsonParseConfig(max_depth=8, max_leaf_tokens=48, max_value_chars=80, numeric_bucket_base=2))

## Control Cell

Сначала выполните `DRY_RUN = True`: он нормализует один shard. Если sample выглядит нормально, поменяйте `DRY_RUN = False` для полного прогона.

`OVERWRITE = False` позволяет продолжить после прерывания: уже готовые `events_normalized_*.parquet` будут пропущены. Если меняли логику нормализации и хотите пересчитать всё, поставьте `OVERWRITE = True`.

In [3]:
DRY_RUN = False
MAX_SHARDS = 1 if DRY_RUN else None
OVERWRITE = True if DRY_RUN else False

print("DRY_RUN:", DRY_RUN)
print("MAX_SHARDS:", MAX_SHARDS)
print("OVERWRITE:", OVERWRITE)

DRY_RUN: False
MAX_SHARDS: None
OVERWRITE: False


In [4]:
manifest = normalize_parquet_shards(
    config=config,
    project_root=PROJECT_ROOT,
    max_shards=MAX_SHARDS,
    overwrite=OVERWRITE,
)

print("num_shards:", manifest["num_shards"])
print("rows:", manifest["rows"])
print("parse_errors:", manifest["parse_errors"])
manifest["shards"][:3]

Normalize shards:   0%|          | 0/74 [00:00<?, ?it/s]

num_shards: 74
rows: 73063802
parse_errors: 6708


[{'shard_idx': 0,
  'input_path': 'data/interim/events_shard_00000.parquet',
  'output_path': 'data/interim/events_normalized_00000.parquet',
  'rows': 1000000,
  'parse_errors': 0,
  'status': 'skipped_existing'},
 {'shard_idx': 1,
  'input_path': 'data/interim/events_shard_00001.parquet',
  'output_path': 'data/interim/events_normalized_00001.parquet',
  'rows': 1000000,
  'parse_errors': 1,
  'status': 'written'},
 {'shard_idx': 2,
  'input_path': 'data/interim/events_shard_00002.parquet',
  'output_path': 'data/interim/events_normalized_00002.parquet',
  'rows': 1000000,
  'parse_errors': 2,
  'status': 'written'}]

In [5]:
import pandas as pd

first_output = PROJECT_ROOT / manifest["shards"][0]["output_path"]
sample = pd.read_parquet(first_output).head(10)

print(first_output)
sample[[
    "user_id",
    "event_name",
    "event_token",
    "event_signature",
    "json_top_keys",
    "json_leaf_paths",
    "attribute_tokens",
    "numeric_bucket_tokens",
    "json_parse_error",
]]

/root/projects/bert4rec/data/interim/events_normalized_00000.parquet


,user_id,event_name,event_token,event_signature,json_top_keys,json_leaf_paths,attribute_tokens,numeric_bucket_tokens,json_parse_error
0,6997871487008262307,BOOT_TIME,event=BOOT_TIME,event=BOOT_TIME|json=Startup,[Startup],"[Startup_elapsed, Startup_index, Startup_total]","[path=Startup_elapsed, path=Startup_index, pat...","[num=Startup_elapsed:zero, num=Startup_index:p...",
1,6997871487008262307,BOOT_TIME,event=BOOT_TIME,event=BOOT_TIME|json=Modules initialization st...,[Modules initialization started],"[Modules initialization started_elapsed, Modul...","[path=Modules initialization started_elapsed, ...",[num=Modules initialization started_elapsed:po...,
2,6997871487008262307,BOOT_TIME,event=BOOT_TIME,event=BOOT_TIME|json=Modules initialization fi...,[Modules initialization finished],"[Modules initialization finished_elapsed, Modu...","[path=Modules initialization finished_elapsed,...",[num=Modules initialization finished_elapsed:p...,
3,6997871487008262307,BOOT_TIME,event=BOOT_TIME,event=BOOT_TIME|json=Game node register reposi...,[Game node register repositories],"[Game node register repositories_elapsed, Game...","[path=Game node register repositories_elapsed,...",[num=Game node register repositories_elapsed:p...,
4,6997871487008262307,BOOT_TIME,event=BOOT_TIME,event=BOOT_TIME|json=Load game data started,[Load game data started],"[Load game data started_elapsed, Load game dat...","[path=Load game data started_elapsed, path=Loa...","[num=Load game data started_elapsed:pos_lt_1, ...",
5,6997871487008262307,BOOT_TIME,event=BOOT_TIME,event=BOOT_TIME|json=Load game data finished,[Load game data finished],"[Load game data finished_elapsed, Load game da...","[path=Load game data finished_elapsed, path=Lo...","[num=Load game data finished_elapsed:pos_lt_1,...",
6,6997871487008262307,BOOT_TIME,event=BOOT_TIME,event=BOOT_TIME|json=InitAB,[InitAB],"[InitAB_elapsed, InitAB_index, InitAB_total]","[path=InitAB_elapsed, path=InitAB_index, path=...","[num=InitAB_elapsed:pos_lt_1, num=InitAB_index...",
7,6997871487008262307,BOOT_TIME,event=BOOT_TIME,event=BOOT_TIME|json=AB Module Start Fetch,[AB Module Start Fetch],"[AB Module Start Fetch_elapsed, AB Module Star...","[path=AB Module Start Fetch_elapsed, path=AB M...","[num=AB Module Start Fetch_elapsed:pos_lt_1, n...",
8,6997871487008262307,BOOT_TIME,event=BOOT_TIME,event=BOOT_TIME|json=Nodes launch,[Nodes launch],"[Nodes launch_elapsed, Nodes launch_index, Nod...","[path=Nodes launch_elapsed, path=Nodes launch_...","[num=Nodes launch_elapsed:pos_lt_1, num=Nodes ...",
9,6997871487008262307,BOOT_TIME,event=BOOT_TIME,event=BOOT_TIME|json=Scene setup,[Scene setup],"[Scene setup_elapsed, Scene setup_index, Scene...","[path=Scene setup_elapsed, path=Scene setup_in...","[num=Scene setup_elapsed:pos_lt_1, num=Scene s...",


In [ ]:
sample[["event_json", "event_json_canonical"]].head(5)

In [ ]:
manifest_path = PROJECT_ROOT / config.manifest_path
print(f"Manifest written to: {manifest_path}")
print(f"First normalized shard: {first_output}")